In [1]:
import pandas as pd
import re
import os

## Delete ALL files contained in 見通し folder

In [2]:
# Delete all files under target_dir recursively, except keep_file
target_dir = r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し"
keep_file = "CombineMitoshi.ipynb"

for root, dirs, files in os.walk(target_dir):
    for fname in files:
        if fname == keep_file:
            continue
        fpath = os.path.join(root, fname)
        try:
            os.remove(fpath)
            print(f"Deleted: {fpath}")
        except Exception as e:
            print(f"Failed: {fpath} -> {e}")

print("Done.")

Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_みらい.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_アグリコ.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_フジ.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_フローラ.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_プラントケア.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_ホールディングス.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_ヤング.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_ランド.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_物流.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_精興園.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_農芸.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\みらい__Result\イノチオみらい㈱.csv
Deleted: C:\Users\2372\OneDri

## Handle downloaded files and Concat them by 会社

In [3]:
import os
import re
import glob
import pandas as pd


# =========================================================
# 1) CONFIG: Add all your folder/output pairs here
# =========================================================
JOBS = [
    # {
    #     "name": "アグリ", #for logging name
    #     "input_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\アグリ_組織",
    #     "save_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリ_組織_Result",
    #     "combined_output": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_アグリ.csv" #must same name as on Fabric
    # },
    {
        "name": "アグリコ",
        "input_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\アグリコ",
        "save_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリコ_Result",
        "combined_output": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_アグリコ.csv"
    },
    {
        "name": "フジ",
        "input_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\フジ",
        "save_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\フジ_Result",
        "combined_output": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_フジ.csv"
    },
    {
        "name": "プラントケア",
        "input_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\プラントケア",
        "save_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\プラントケア",
        "combined_output": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_プラントケア.csv"
    },
    {
        "name": "フローラ",
        "input_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\フローラ_組織",
        "save_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\フローラ__Result",
        "combined_output": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_フローラ.csv"
    },
    {
        "name": "ホールディングス",
        "input_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\ホールディングス_組織",
        "save_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\ホールディングス_組織_Result",
        "combined_output": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_ホールディングス.csv"
    },
    {
        "name": "みらい",
        "input_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\みらい_組織",
        "save_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\みらい__Result",
        "combined_output": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_みらい.csv"
    },
    {
        "name": "ヤング",
        "input_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\ヤング",
        "save_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\ヤング__Result",
        "combined_output": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_ヤング.csv"
    },
    {
        "name": "ランド",
        "input_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\ランド",
        "save_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\ランド__Result",
        "combined_output": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_ランド.csv"
    },
    {
        "name": "物流",
        "input_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\物流",
        "save_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\物流",
        "combined_output": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_物流.csv"
    },
    {
        "name": "精興園",
        "input_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\精興園_組織",
        "save_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\精興園__Result",
        "combined_output": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_精興園.csv"
    },
    {
        "name": "農芸",
        "input_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\農芸_組織",
        "save_dir": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\農芸__Result",
        "combined_output": r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_農芸.csv"
    },
]


# =========================================================
# 2) Utility functions
# =========================================================
def _norm(s):
    """列名の揺れに強くする簡易正規化（全角/半角・空白除去・小文字化）"""
    if s is None:
        return ""
    s = str(s)
    s = s.replace("\u3000", "").replace(" ", "")
    s = s.replace("Ｍ", "M").replace("Ｐ", "P").replace("／", "/")
    return s.lower().strip()


def sanitize_filename(name):
    """Windowsで使えない文字を除去"""
    if not isinstance(name, str):
        name = str(name)
    return re.sub(r'[\\/:*?"<>|]', "", name).strip()


def clean_numeric_series(s):
    return (
        s.astype(str)
         .str.replace(",", "", regex=False)
         .str.replace("¥", "", regex=False)
         .str.replace("%", "", regex=False)
         .str.replace("\u3000", "", regex=False)
         .str.strip()
    )


def find_subject_col(df_cols):
    # サブヘッダが「科目」
    subject_candidates = [c for c in df_cols if _norm(c[1]) == _norm("科目")]
    if subject_candidates:
        return subject_candidates[0]

    # 文字列に「科目」を含む
    subject_candidates = [c for c in df_cols if "科目" in "".join(map(str, c))]
    return subject_candidates[0] if subject_candidates else None


def find_mitoshi_col(df_cols):
    """
    「見通し」を優先して探す。
    ・(top='年度', sub='見通し') を最優先
    ・それがなければ、サブヘッダが '見通し' の最初の列
    """
    strict = [c for c in df_cols if _norm(c[0]) == _norm("年度") and _norm(c[1]) == _norm("見通し")]
    if strict:
        return strict[0]

    loose = [c for c in df_cols if _norm(c[1]) == _norm("見通し")]
    return loose[0] if loose else None


def find_under_same_top(df_cols, top_label, sub_name_candidates):
    """
    上段ヘッダが top_label と一致し、
    サブヘッダが候補に一致する列を返す。
    """
    sub_keys = {_norm(s) for s in sub_name_candidates}

    for c in df_cols:
        if _norm(c[0]) == _norm(top_label) and _norm(c[1]) in sub_keys:
            return c

    # 緩い判定（例: "MP比" in "MP比(%)"）
    for c in df_cols:
        if _norm(c[0]) == _norm(top_label):
            ns = _norm(c[1])
            for k in sub_keys:
                if k and k in ns:
                    return c
    return None


# =========================================================
# 3) Process one source CSV -> one separated organization CSV
# =========================================================
def process_one_file(path, save_dir):
    try:
        raw = pd.read_csv(path, encoding="utf-8-sig", header=None)

        if raw.empty or len(raw) < 6:
            print(f"SKIP (empty or too few rows): {path}")
            return None

        # --- 1) 組織情報抽出（2行目） ---
        row2 = raw.iloc[1].astype(str)
        first_non_empty = next((v for v in row2 if v and v.lower() != "nan"), "")

        org_line = (
            first_non_empty
            .replace("\u3000", "")  # 全角スペース除去
            .strip()
            .replace("＿", "_")     # 全角アンダースコア→半角
        )

        # "230H60200_総務課" のような形式を分割
        org_code, org_name = "", ""
        m = re.match(r"^\s*([0-9A-Za-z]+)_(.+?)\s*$", org_line)
        if m:
            org_code, org_name = m.group(1), m.group(2).strip()
        else:
            if "_" in org_line:
                parts = org_line.split("_", 1)
                org_code, org_name = parts[0].strip(), parts[1].strip()
            else:
                org_name = org_line.strip()

        # --- 2) ヘッダ構築 ---
        if len(raw) < 5:
            print(f"SKIP (header rows missing): {path}")
            return None

        top = raw.iloc[3].astype(str).str.strip().replace({"nan": ""}).str.replace(" ", "", regex=False)
        sub = raw.iloc[4].astype(str).str.strip().replace({"nan": ""}).str.replace(" ", "", regex=False)

        filled_top = []
        last = ""
        for val in top:
            if val:
                last = val
                filled_top.append(val)
            else:
                filled_top.append(last)

        multi_cols = pd.MultiIndex.from_tuples(list(zip(filled_top, sub)))
        data = raw.iloc[5:].reset_index(drop=True)

        # 列数ズレ対策
        if data.shape[1] != len(multi_cols):
            min_cols = min(data.shape[1], len(multi_cols))
            data = data.iloc[:, :min_cols]
            multi_cols = multi_cols[:min_cols]

        data.columns = multi_cols

        # 科目列
        subject_col = find_subject_col(data.columns)
        if subject_col is None:
            print(f"SKIP (科目列が見つからない): {path}")
            return None

        # 年度_見通し列
        col_mitoshi = find_mitoshi_col(data.columns)
        if col_mitoshi is None:
            print(f"SKIP (年度_見通し列が見つからない): {path}")
            return None

        the_top = col_mitoshi[0]

        mp_candidates = ["M/P", "MP", "Ｍ／Ｐ"]
        mp_ratio_candidates = ["M/P比", "M/P比(%)", "MP比", "MP比(%)"]
        yoy_candidates = ["前年比", "前年比(%)"]

        col_mp = find_under_same_top(data.columns, the_top, mp_candidates)
        col_mp_ratio = find_under_same_top(data.columns, the_top, mp_ratio_candidates)
        col_yoy = find_under_same_top(data.columns, the_top, yoy_candidates)

        # 使用列
        use_cols = [subject_col, col_mitoshi]
        if col_mp is not None:
            use_cols.append(col_mp)
        if col_mp_ratio is not None:
            use_cols.append(col_mp_ratio)
        if col_yoy is not None:
            use_cols.append(col_yoy)

        out = data[use_cols].copy()

        # 列名フラット化
        rename_map = {
            subject_col: "科目",
            col_mitoshi: "年度_見通し"
        }
        if col_mp is not None:
            rename_map[col_mp] = "MP"
        if col_mp_ratio is not None:
            rename_map[col_mp_ratio] = "MP比"
        if col_yoy is not None:
            rename_map[col_yoy] = "前年比"

        out.columns = [rename_map.get(c, "_".join(map(str, c))) for c in out.columns]

        # --- 3) クリーニング ---
        out["科目"] = (
            out["科目"].astype(str)
            .str.replace("\u3000", "", regex=False)
            .str.strip()
        )
        out = out[out["科目"].notna() & (out["科目"] != "") & (out["科目"].str.lower() != "nan")]

        for col in ["年度_見通し", "MP", "MP比", "前年比"]:
            if col in out.columns:
                out[col] = clean_numeric_series(out[col])
                out[col] = pd.to_numeric(out[col], errors="coerce")

        # --- 4) 組織コード/名 ---
        org_code2, org_name2 = "", ""
        if "_" in org_line:
            org_code2, org_name2 = org_line.split("_", 1)
            org_code2 = org_code2.strip()
            org_name2 = org_name2.strip()
        else:
            org_name2 = org_line.strip()

        org_name_sanitized = sanitize_filename(org_name if org_name else org_name2)

        out["組織コード"] = org_code if org_code else org_code2
        out["組織名"] = org_name if org_name else org_name2

        # 必須列がなければ追加
        for need in ["年度_見通し", "MP", "MP比", "前年比"]:
            if need not in out.columns:
                out[need] = pd.NA

        out = out[["組織コード", "組織名", "科目", "年度_見通し", "MP", "MP比", "前年比"]]

        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, f"{org_name_sanitized}.csv")
        out.to_csv(save_path, index=False, sep="\t", encoding="utf-8-sig")
        print(f"SAVED: {save_path}")
        return save_path

    except Exception as e:
        print(f"ERROR: {path} -> {e}")
        return None


# =========================================================
# 4) Process all source CSVs in one folder
# =========================================================
def process_folder(input_dir, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    if not os.path.exists(input_dir):
        print(f"SKIP: input_dir does not exist -> {input_dir}")
        return []

    saved_files = []
    for fname in os.listdir(input_dir):
        if not fname.lower().endswith(".csv"):
            continue

        fpath = os.path.join(input_dir, fname)
        if os.path.isfile(fpath):
            result = process_one_file(fpath, save_dir)
            if result:
                saved_files.append(result)

    print(f"Processed {len(saved_files)} separated files in: {input_dir}")
    return saved_files


# =========================================================
# 5) Combine separated files in one save_dir
# =========================================================
def combine_folder(save_dir, combined_output):
    file_list = glob.glob(os.path.join(save_dir, "*.csv"))

    if not file_list:
        print(f"SKIP: No separated CSV files found in {save_dir}")
        return None

    float_columns = ["年度_見通し", "MP", "MP比", "前年比"]
    dfs = []

    for f in file_list:
        try:
            with open(f, "r", encoding="utf-8-sig", errors="ignore") as file:
                content = file.read(2000)  # 最初の一部だけで十分
                if "<!DOCTYPE HTML" in content.upper():
                    print(f"Skip HTML file: {f}")
                    continue

            df = pd.read_csv(
                f,
                encoding="utf-8-sig",
                delimiter="\t",
                on_bad_lines="skip",
                dtype={"組織コード": str}
            )

            for col in float_columns:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col].replace("-", 0), errors="coerce")

            dfs.append(df)

        except Exception as e:
            print(f"ERROR reading {f}: {e}")

    if not dfs:
        print(f"SKIP: No valid dataframes to combine in {save_dir}")
        return None

    df_all = pd.concat(dfs, ignore_index=True)

    output_dir = os.path.dirname(combined_output)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    df_all.to_csv(combined_output, index=False, sep="\t", encoding="utf-8-sig")

    print(f"COMBINED SAVED: {combined_output}")
    print(f"Read {len(file_list)} files from {save_dir}")
    print(f"Total rows: {len(df_all)}")

    return df_all


# =========================================================
# 6) Run one job: separate + combine
# =========================================================
def run_job(job):
    name = job["name"]
    input_dir = job["input_dir"]
    save_dir = job["save_dir"]
    combined_output = job["combined_output"]

    print("\n" + "=" * 80)
    print(f"START JOB: {name}")
    print(f"Input : {input_dir}")
    print(f"Save  : {save_dir}")
    print(f"Output: {combined_output}")

    process_folder(input_dir, save_dir)
    combine_folder(save_dir, combined_output)

    print(f"END JOB: {name}")
    print("=" * 80)


# =========================================================
# 7) Run all jobs
# =========================================================
def main():
    for job in JOBS:
        run_job(job)


if __name__ == "__main__":
    main()


START JOB: アグリコ
Input : C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\アグリコ
Save  : C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリコ_Result
Output: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_アグリコ.csv
SAVED: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリコ_Result\㈱アグリコ.csv
SAVED: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリコ_Result\本社収支.csv
SAVED: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリコ_Result\アグリコ.csv
SAVED: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリコ_Result\営業部.csv
SAVED: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\SeperatedFiles\アグリコ_Result\園芸センター.csv
Processed 5 separated files in: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\アグリコ
COMBINED SAVED: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\見通し\見通し_アグリコ.csv
Read 5 files from C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採